# 05. Colab GPU 모델 비교

기준일 `t`까지 확인 가능한 정보로 `t+1`~`t+7`의 일별 판매량을 예측합니다. 현재 LightGBM과 CatBoost를 같은 전체 이력 Feature 및 최근 28개 검증 기준일에서 비교합니다. 이후 딥러닝 결과도 이 노트북의 최종 비교 표에 추가합니다.

My Drive의 저장소와 원본 데이터는 Git 업로드 대상이 아닙니다. 전체 학습은 시간이 오래 걸리므로 필요한 셀만 실행하세요.


## Library


In [ ]:
import gc
import json
import os
import subprocess
import sys
from pathlib import Path
from time import sleep

from google.colab import drive

drive.mount("/content/drive")
REPO_URL = "https://github.com/hyun0415/retail-demand-forecasting.git"
REPO = Path("/content/drive/MyDrive/retail-demand-forecasting")
if not REPO.parent.is_dir():
    raise FileNotFoundError(f"My Drive 경로를 찾지 못했습니다: {REPO.parent}")
if REPO.exists() and not (REPO / ".git").is_dir():
    raise RuntimeError(f"저장소 경로에 .git이 없습니다: {REPO}")
if not REPO.exists():
    cloned = subprocess.run(["git", "clone", REPO_URL, str(REPO)], capture_output=True, text=True)
    if cloned.returncode:
        raise RuntimeError(f"git clone 실패 (exit {cloned.returncode}):\n{cloned.stderr.strip()}")
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
os.chdir(REPO)
print("Repository:", REPO)
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


### 공통 패키지 설치

CatBoost 진단에도 필요한 패키지를 설치합니다. Colab GPU를 선택해야 합니다. 아래 LightGBM CUDA 빌드는 CatBoost 진단에서는 건너뜁니다.


In [ ]:
subprocess.run(["nvidia-smi"], check=True)
required = (REPO / "requirements.txt").read_text(encoding="utf-8").splitlines()
packages = [line for line in required if line.strip()]
subprocess.run([sys.executable, "-m", "pip", "install", *packages, "-e", str(REPO)], check=True)


### LightGBM CUDA 빌드: 전체 ML 실험에만 사용

CatBoost 단기 진단에서는 이 셀을 건너뜁니다. 전체 LightGBM GPU 학습을 다시 실행할 때는 Library import 셀보다 먼저 실행하세요.


In [ ]:
subprocess.run(["nvcc", "--version"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "lightgbm"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--no-cache-dir", "--no-binary=lightgbm", "--no-deps",
    "--config-settings=cmake.define.USE_CUDA=ON",
    "--config-settings=cmake.define.CMAKE_CUDA_ARCHITECTURES=native",
    "lightgbm==4.7.0",
], check=True)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from tqdm.auto import tqdm

sys.path.insert(0, str(REPO / "src"))
from retail_forecast.config import load_config
from retail_forecast.metrics import regression_metrics
from retail_forecast.pipeline import feature_cache_path, run_experiment

plt.style.use("seaborn-v0_8-whitegrid")


## Utils

결과 표를 만드는 보조 함수만 이곳에 둡니다. Feature 생성과 모델 학습은 `src/retail_forecast`의 함수를 사용합니다.


In [ ]:
def load_model_result(result_dir, model_name, commit):
    metrics_path = result_dir / f"metrics_{model_name}.json"
    summary_path = result_dir / f"decision_summary_{model_name}.csv"
    horizon_path = result_dir / f"metrics_by_horizon_{model_name}.csv"
    missing = [path for path in (metrics_path, summary_path, horizon_path) if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"모델 실행 결과가 없습니다: {missing[0]}")

    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    summary = pd.read_csv(summary_path, usecols=["actual_sum_1_7", "prediction_sum_1_7"])
    if summary.isna().any().any():
        raise ValueError(f"{model_name}: 7일 합계에 결측값이 있습니다")
    weekly = regression_metrics(
        summary["actual_sum_1_7"].to_numpy(),
        summary["prediction_sum_1_7"].to_numpy(),
    )
    row = {
        "model": model_name, "training_scope": "all_available_origins",
        "git_commit": commit, "validation_origin_start": metrics["validation_origin_start"],
        "train_rows": metrics["train_rows"], "validation_rows": metrics["validation_rows"],
        **{name: metrics[name] for name in ("rmsle", "mae", "wape", "bias")},
        **{f"sum_1_7_{name}": value for name, value in weekly.items()},
    }
    horizon = pd.read_csv(horizon_path)
    horizon.insert(0, "model", model_name)
    return row, horizon


def build_comparison_tables(result_dir, commit):
    rows, horizons = [], []
    for model_name in ("lightgbm", "catboost"):
        row, horizon = load_model_result(result_dir, model_name, commit)
        rows.append(row)
        horizons.append(horizon)
    return pd.DataFrame(rows), pd.concat(horizons, ignore_index=True)


## 비교 모델

| 구분 | 모델 | 역할 | 상태 |
|---|---|---|---|
| ML | LightGBM, CatBoost | 공통 Feature 기반 예측 | 현재 실행 가능 |
| DL | [N-HiTS](https://nixtlaverse.nixtla.io/neuralforecast/models.nhits.html) | MLP 계열의 다중 해상도 예측 | 설계 단계 |
| DL | [TCN](https://nixtlaverse.nixtla.io/neuralforecast/models.tcn.html) | CNN 계열의 시간 패턴 예측 | 설계 단계 |
| DL | [TFT](https://nixtlaverse.nixtla.io/neuralforecast/models.tft.html) | 순환 계층과 attention 기반 예측 | 설계 단계 |

DL 세 모델은 NeuralForecast로 구현할 예정입니다. 현재 노트북에는 DL 학습 코드나 결과가 없으며, 같은 검증 기준일·지표로 평가한 뒤 이 비교표에 추가합니다. DL 입력은 원본 일별 패널의 과거 시퀀스를 사용합니다.


## 1. GPU 및 진행 표시 확인

진행 표시 시험은 CatBoost 진단 전에 실행할 수 있습니다. 아래 GPU 시험 학습 셀은 LightGBM CUDA 빌드를 실행한 전체 ML 경로에서만 사용하세요.


In [ ]:
for _ in tqdm(range(10), desc="진행 표시 시험", unit="칸"):
    sleep(0.05)


## CatBoost 단기 진단

기존 전체 이력 Feature 캐시에서 검증 직전 180개 학습 기준일만 읽습니다. **검증 기준일은 기존과 같은 최근 28개**이며 학습 정답 날짜는 검증 시작일보다 앞서야 합니다. `MAE`와 `RMSE`를 각각 GPU에서 300회 학습하며, 원래 모델·결과 파일은 건드리지 않습니다. 결과는 `outputs/diagnostics/catboost_t180_i300`에 저장됩니다.

**실행 셀:** Library의 저장소 연결·공통 패키지 설치·import, Utils, 진행 표시 시험을 실행한 뒤 아래 셀을 실행하세요. LightGBM CUDA 빌드, GPU 시험 학습, 전체 Feature 생성, 전체 ML 학습 셀은 건너뜁니다. Drive 저장소에 기존 전체 이력 Feature 캐시가 있어야 합니다.

표에서 `prediction_max`, `high_sales_prediction_mean`, `bias`, `wape`를 비교합니다. 뒤의 **결과 표 저장** 섹션은 기존 전체 ML 실험을 위한 것이므로 이 진단에서는 실행하지 않습니다.


In [ ]:
diagnostic_config = load_config(REPO / "configs/full_catboost_gpu.toml")
if not feature_cache_path(diagnostic_config).exists():
    raise FileNotFoundError("전체 이력 Feature 캐시가 없습니다. 기존 Drive 저장소 경로를 확인하세요.")
subprocess.run([
    sys.executable, "scripts/diagnose_catboost.py",
    "--config", "configs/full_catboost_gpu.toml",
    "--train-origin-days", "180",
    "--iterations", "300",
], cwd=REPO, check=True)
diagnostic_dir = REPO / "outputs/diagnostics/catboost_t180_i300"
display(pd.read_csv(diagnostic_dir / "comparison.csv").round(4))


### 전체 ML 재실행 시 GPU 시험 학습

LightGBM CUDA 빌드를 실행한 경우에만 아래 셀을 실행합니다. CatBoost 진단에는 필요하지 않습니다.


In [ ]:
toy_x = pd.DataFrame(np.arange(240, dtype=np.float32).reshape(80, 3))
toy_x["family"] = pd.Categorical(["A", "B"] * 40)
toy_y = np.arange(80, dtype=np.float32)
LGBMRegressor(
    n_estimators=2, min_child_samples=2, objective="regression_l1",
    device_type="cuda", verbosity=-1,
).fit(toy_x, toy_y)
toy_x["family"] = toy_x["family"].astype(str)
CatBoostRegressor(
    iterations=2, loss_function="MAE", task_type="GPU", devices="0", verbose=False,
).fit(toy_x, toy_y, cat_features=["family"])
print("두 모델의 GPU 시험 학습이 완료됐습니다.")

## 2. Kaggle 원본과 전체 이력 Feature

`train.csv`, `stores.csv`, `holidays_events.csv`, `transactions.csv`를 clone한 저장소의 `data/raw`에 넣으세요. 원본과 생성 파일은 Git 추적 대상이 아닙니다. Feature는 28일 단위로 나누어 Parquet에 저장하며, 이미 완성된 캐시는 다시 만들지 않습니다.

In [ ]:
required_files = ("train.csv", "stores.csv", "holidays_events.csv", "transactions.csv")
missing = [name for name in required_files if not (REPO / "data/raw" / name).is_file()]
if missing:
    raise FileNotFoundError(f"data/raw에 다음 파일을 넣으세요: {', '.join(missing)}")
subprocess.run([
    sys.executable, "scripts/build_full_features.py",
    "--config", "configs/full_lightgbm_gpu.toml",
    "--chunk-days", "28",
], cwd=REPO, check=True)

## 3. 모델별 전체 이력 학습

두 설정은 같은 Feature 캐시와 검증 구간을 사용합니다. LightGBM 실행이 끝난 뒤 CatBoost를 실행하세요. 각 실행은 `models/full_history`, `outputs/full_history`에 저장되어 기존 365일 실험 결과를 덮어쓰지 않습니다.

In [ ]:
lightgbm_config = load_config(REPO / "configs/full_lightgbm_gpu.toml")
if not feature_cache_path(lightgbm_config).exists():
    raise FileNotFoundError("전체 이력 Feature 캐시가 없습니다. 이전 셀을 먼저 실행하세요.")
lightgbm_result = run_experiment(lightgbm_config, progress=True)
print(lightgbm_result)

In [ ]:
gc.collect()
catboost_config = load_config(REPO / "configs/full_catboost_gpu.toml")
if feature_cache_path(catboost_config) != feature_cache_path(lightgbm_config):
    raise ValueError("두 모델의 Feature 캐시 경로가 다릅니다.")
catboost_result = run_experiment(catboost_config, progress=True)
print(catboost_result)

## 4. 결과 표 저장

전체 일별 예측, horizon별 예측, 발주용 7일 합계의 오차를 별도 표로 저장합니다. 두 모델의 결과 파일이 모두 있을 때만 비교표를 만듭니다.

In [ ]:
result_dir = REPO / "outputs/full_history"
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
comparison, by_horizon = build_comparison_tables(result_dir, commit)
comparison.to_csv(result_dir / "comparison_full_history.csv", index=False)
by_horizon.to_csv(result_dir / "comparison_by_horizon_full_history.csv", index=False)
display(comparison)
display(by_horizon)
print("저장 위치:", result_dir)


## 모델 비교 시각화


In [ ]:
plot_data = comparison.set_index("model")[["rmsle", "mae", "wape"]]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, metric in zip(axes, ("rmsle", "mae", "wape")):
    plot_data[metric].sort_values().plot.barh(ax=ax, title=metric.upper())
    ax.set(xlabel="낮을수록 좋음", ylabel="")
plt.tight_layout()


## 모델 선택과 해석

RMSLE, MAE, WAPE, Bias를 함께 보고 예측 시차별 성능과 7일 합계 오류도 확인합니다. 성능 차이가 작다면 학습 시간과 설명 편의성을 고려합니다. 상품군·매장별 오류는 `06_error_analysis.ipynb`에서 분석합니다.

딥러닝 모델 결과는 같은 검증 기준일과 지표로 저장한 뒤 이 비교 표에 연결합니다. 아직 실행하지 않은 모델의 수치는 표시하지 않습니다.


### 해석과 주의사항

- 전체 이력은 *모든 기준일*을 사용한다는 뜻입니다. 마지막 28개 기준일의 목표 판매량은 학습에서 제외합니다.
- 이전 365일 결과와 학습 기간과 Feature 조건이 달라 숫자를 직접 비교하면 안 됩니다. 이번 설정은 예측 시점에 알 수 없는 지진·돌발 이벤트 변수를 제외합니다.
- 목표일 프로모션은 **7일 앞까지 계획값을 실제로 확보할 수 있을 때만** 유효합니다. 계획값이 없다면 두 설정에서 `target_onpromotion`을 제외하고 Feature 캐시를 그대로 사용해 재학습해야 합니다.
- `sales`와 `current_transactions`는 기준일 영업 종료 후 예측한다는 전제입니다. 영업 중 발주라면 이 변수도 제외해야 합니다.
- 전체 이력 표는 메모리를 많이 사용합니다. Colab의 GPU와 RAM 할당은 변할 수 있고, My Drive I/O는 느릴 수 있습니다. 런타임이 끊겨도 완성된 Parquet과 결과 파일은 Drive에 남습니다. `.building` 폴더가 남았다면 생성이 중단된 것이므로 내용 확인 후 수동으로 정리하세요.